In [4]:
import hashlib
import time


# ===== Block Class =====
class Block:
    def __init__(self, index, voter_id, vote, previous_hash):
        self.index = index
        self.timestamp = time.time()
        self.voter_id = voter_id
        self.vote = vote
        self.previous_hash = previous_hash
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        value = (
            str(self.index)
            + str(self.timestamp)
            + str(self.voter_id)
            + str(self.vote)
            + str(self.previous_hash)
        )
        return hashlib.sha256(value.encode()).hexdigest()


# ===== Blockchain Class =====
class Blockchain:
    def __init__(self):
        self.chain = [self.create_genesis_block()]
        self.voters = set()
        self.candidates = set()

    def create_genesis_block(self):
        return Block(0, "Genesis", "None", "0")

    def get_latest_block(self):
        return self.chain[-1]

    # Add a candidate
    def add_candidate(self, candidate_name):
        if candidate_name in self.candidates:
            print("Candidate already exists")
            return False

        self.candidates.add(candidate_name)
        print(f"Candidate added: {candidate_name}")
        return True

    # Cast a vote
    def cast_vote(self, voter_id, candidate_name):
        if candidate_name not in self.candidates:
            print("Candidate not found")
            return False

        if voter_id in self.voters:
            print("Voter already voted")
            return False

        latest_block = self.get_latest_block()

        new_block = Block(
            len(self.chain),
            voter_id,
            candidate_name,
            latest_block.hash
        )

        self.chain.append(new_block)
        self.voters.add(voter_id)

        print(
            f"Vote cast successfully for {candidate_name} "
            f"by {voter_id}"
        )
        return True

    # Validate the blockchain
    def is_chain_valid(self):
        # Check the Genesis block
        genesis = self.chain[0]

        if genesis.hash != genesis.calculate_hash():
            return False

        for i in range(1, len(self.chain)):
            current = self.chain[i]
            prev = self.chain[i - 1]

            # Check the current block's hash
            if current.hash != current.calculate_hash():
                return False

            # Check the connection with the previous block
            if current.previous_hash != prev.hash:
                return False

        return True

    # Count votes
    def count_votes(self):
        results = {
            candidate: 0
            for candidate in self.candidates
        }

        for block in self.chain[1:]:
            if block.vote in results:
                results[block.vote] += 1

        return results


# ===== Main Demo =====
if __name__ == "__main__":

    blockchain = Blockchain()

    # Add candidates
    blockchain.add_candidate("Candidate A")
    blockchain.add_candidate("Candidate B")

    print("\n--- Voting Process ---")

    # Cast votes
    blockchain.cast_vote("voter1", "Candidate A")
    blockchain.cast_vote("voter2", "Candidate B")

    # Duplicate vote (should fail)
    blockchain.cast_vote("voter1", "Candidate A")

    # Validate blockchain
    print("\nBlockchain valid:", blockchain.is_chain_valid())

    # Display vote counts
    print("\nVote Counts:")

    for candidate, count in blockchain.count_votes().items():
        print(f"{candidate}: {count}")

    # Display blockchain contents
    print("\nBlockchain Contents:")

    for block in blockchain.chain:
        print("-" * 40)
        print(f"Index: {block.index}")
        print(f"Timestamp: {block.timestamp}")
        print(f"Voter ID: {block.voter_id}")
        print(f"Vote: {block.vote}")
        print(f"Previous Hash: {block.previous_hash[:10]}...")
        print(f"Hash: {block.hash[:10]}...")

    print("-" * 40)


Candidate added: Candidate A
Candidate added: Candidate B

--- Voting Process ---
Vote cast successfully for Candidate A by voter1
Vote cast successfully for Candidate B by voter2
Voter already voted

Blockchain valid: True

Vote Counts:
Candidate A: 1
Candidate B: 1

Blockchain Contents:
----------------------------------------
Index: 0
Timestamp: 1793361948.3813925
Voter ID: Genesis
Vote: None
Previous Hash: 0...
Hash: ba18cdf301...
----------------------------------------
Index: 1
Timestamp: 1793361948.3815541
Voter ID: voter1
Vote: Candidate A
Previous Hash: ba18cdf301...
Hash: fec12fef49...
----------------------------------------
Index: 2
Timestamp: 1793361948.3815737
Voter ID: voter2
Vote: Candidate B
Previous Hash: fec12fef49...
Hash: 7d252b3f29...
----------------------------------------


In [ ]:

import hashlib
import json
import time


# Voter Entity
class Voter:
    def __init__(self, voter_id, name):
        self.voter_id = voter_id
        self.name = name
        self.has_voted = False


# Candidate Entity
class Candidate:
    def __init__(self, candidate_id, name):
        self.candidate_id = candidate_id
        self.name = name


# Block Class
class Block:
    def __init__(self, index, transaction, previous_hash):
        self.index = index
        self.timestamp = time.time()
        self.transaction = transaction
        self.previous_hash = previous_hash
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        block_data = {
            "index": self.index,
            "timestamp": self.timestamp,
            "transaction": self.transaction,
            "previous_hash": self.previous_hash
        }

        encoded_data = json.dumps(
            block_data, sort_keys=True
        ).encode("utf-8")

        return hashlib.sha256(encoded_data).hexdigest()


# Voting Blockchain Class
class VotingBlockchain:
    def __init__(self):
        self.voters = {}
        self.candidates = {}

        genesis_transaction = {
            "type": "GENESIS",
            "message": "First block of the voting blockchain"
        }

        self.chain = [
            Block(0, genesis_transaction, "0")
        ]

    # Add Candidate
    def add_candidate(self, candidate_id, name):
        if not candidate_id or not name:
            print("Candidate ID and name cannot be empty.")
            return False

        if candidate_id in self.candidates:
            print("Error: Candidate ID already exists.")
            return False

        self.candidates[candidate_id] = Candidate(
            candidate_id, name
        )

        print("Candidate added successfully.")
        return True

    # Add Voter
    def add_voter(self, voter_id, name):
        if not voter_id or not name:
            print("Voter ID and name cannot be empty.")
            return False

        if voter_id in self.voters:
            print("Error: Voter ID already exists.")
            return False

        self.voters[voter_id] = Voter(voter_id, name)

        print("Voter added successfully.")
        return True

    # Cast Vote
    def cast_vote(self, voter_id, candidate_id):
        voter = self.voters.get(voter_id)
        candidate = self.candidates.get(candidate_id)

        if voter is None:
            print("Error: Voter not found.")
            return False

        if candidate is None:
            print("Error: Candidate not found.")
            return False

        if voter.has_voted:
            print("Error: Voter has already voted.")
            return False

        transaction = {
            "type": "VOTE",
            "voter_id": voter.voter_id,
            "voter_name": voter.name,
            "candidate_id": candidate.candidate_id,
            "candidate_name": candidate.name
        }

        previous_block = self.chain[-1]

        new_block = Block(
            len(self.chain),
            transaction,
            previous_block.hash
        )

        self.chain.append(new_block)
        voter.has_voted = True

        print("Vote cast successfully.")
        return True

    # Print Blockchain
    def print_blockchain(self):
        print("\n========== BLOCKCHAIN CONTENTS ==========")

        for block in self.chain:
            print("\n" + "-" * 45)
            print("Block Index:", block.index)
            print("Timestamp:", time.ctime(block.timestamp))
            print("Transaction:")
            print(json.dumps(block.transaction, indent=4))
            print("Previous Hash:", block.previous_hash)
            print("Block Hash:", block.hash)

        print("-" * 45)

    # Validate Blockchain
    def validate_chain(self):
        if not self.chain:
            return False

        genesis = self.chain[0]

        if genesis.previous_hash != "0":
            return False

        if genesis.hash != genesis.calculate_hash():
            return False

        seen_voters = set()

        for i in range(1, len(self.chain)):
            current = self.chain[i]
            previous = self.chain[i - 1]

            if current.hash != current.calculate_hash():
                return False

            if current.previous_hash != previous.hash:
                return False

            transaction = current.transaction

            if transaction.get("type") != "VOTE":
                return False

            voter_id = transaction.get("voter_id")
            candidate_id = transaction.get("candidate_id")

            if not voter_id or voter_id in seen_voters:
                return False

            if not candidate_id:
                return False

            seen_voters.add(voter_id)

        return True


# Menu-Driven Application
def main():
    voting_system = VotingBlockchain()

    while True:
        print("\n===== VOTING MANAGEMENT SYSTEM =====")
        print("1. Add Candidate")
        print("2. Add Voter")
        print("3. Cast Vote")
        print("4. Print Blockchain")
        print("5. Validate Chain")
        print("6. Exit")

        choice = input("Enter your choice (1-6): ").strip()

        if choice == "1":
            candidate_id = input("Enter candidate ID: ").strip()
            name = input("Enter candidate name: ").strip()

            voting_system.add_candidate(candidate_id, name)

        elif choice == "2":
            voter_id = input("Enter voter ID: ").strip()
            name = input("Enter voter name: ").strip()

            voting_system.add_voter(voter_id, name)

        elif choice == "3":
            voter_id = input("Enter voter ID: ").strip()
            candidate_id = input("Enter candidate ID: ").strip()

            voting_system.cast_vote(voter_id, candidate_id)

        elif choice == "4":
            voting_system.print_blockchain()

        elif choice == "5":
            if voting_system.validate_chain():
                print("Blockchain is valid.")
            else:
                print("Blockchain is invalid or tampered with.")

        elif choice == "6":
            print("Thank you for using the system.")
            break

        else:
            print("Invalid choice. Please enter 1 to 6.")


if __name__ == "__main__":
    main()



===== VOTING MANAGEMENT SYSTEM =====
1. Add Candidate
2. Add Voter
3. Cast Vote
4. Print Blockchain
5. Validate Chain
6. Exit
Enter your choice (1-6): 1
Enter candidate ID: EP20
Enter candidate name: Yash Rathode
Candidate added successfully.

===== VOTING MANAGEMENT SYSTEM =====
1. Add Candidate
2. Add Voter
3. Cast Vote
4. Print Blockchain
5. Validate Chain
6. Exit
